In [0]:
# Create own custom schema to improve performance and reduce memory usage
from pyspark.sql.types import *

schema = StructType([
    StructField("Country", StringType(), True),
    StructField("Region", StringType(), True),
    StructField("Date", StringType(), True),
    StructField("Kilotons of Co2", DoubleType(), True),
    StructField("Metric Tons Per Capita", DoubleType(), True)
])

# Then, read the csv file using the custom schema and convert to a Spark DataFrame

file_location = "/Volumes/databrickslinkedin/default/fortesting/Carbon_(CO2)_Emissions_by_Country.csv"
df = spark.read.csv(file_location, header=True, schema=schema)
display(df.limit(5))

In [0]:
# Since we changed BELOW the format of the file to Parquet, we need to change the file location and read the file using the parquet format
from pyspark.sql.types import *

schema = StructType([
    StructField("Country", StringType(), True),
    StructField("Region", StringType(), True),
    StructField("Date", StringType(), True),
    StructField("Kilotons of Co2", DoubleType(), True),
    StructField("Metric Tons Per Capita", DoubleType(), True)
])

# Then, read the Parket file using the parquet format

file_location = "/Volumes/databrickslinkedin/default/fortesting/part-00000-tid-9187313589586967629-03c6cdec-55c4-419f-ba44-984b83f31c6b-4-1.c000.snappy.parquet"
df = spark.read.parquet(file_location, header=True, schema=schema)
display(df.limit(5))

In [0]:
# GroupBy and aggregation
# ascending=True (default) sorts from smallest to largest, ascending=False sorts from largest to smallest
from pyspark.sql.functions import sum as _sum

top10Emitters = display(
    df.groupBy("Country")
      .agg(_sum("Kilotons of Co2").alias("Total_Co2"))
      .orderBy("Total_Co2", ascending=False).limit(10)
)

In [0]:
# Save the dataframe as a parquet file

df.write.mode("overwrite").format("parquet").save("/Volumes/databrickslinkedin/default/fortesting/")



In [0]:
# Check the parket file created previously
# https://learn.microsoft.com/en-us/azure/databricks/query/formats/parquet

df = spark.read.parquet("/Volumes/databrickslinkedin/default/fortesting/")
display(df)

In [0]:
# The client is only interested in a specific country ---> Filter the dataframe
df.filter("Country = 'Netherlands'").display()

In [0]:
# The client is only interested in The Netherlands when date is 01-01-2019.
df.filter("Country = 'Netherlands' AND Date = '01-01-2019'").select("Country", "Date", "Kilotons of Co2").display()

In [0]:
# Final step: Load the parquet file into a delta lake table
# https://learn.microsoft.com/en-us/azure/databricks/delta/

# Rename ALL columns with spaces or invalid characters, Delta Lake Tables do not acccept spaces or invalid characters.
df_parquet = df_parquet.withColumnRenamed("Kilotons of Co2", "Kilotons_of_Co2") \
    .withColumnRenamed("Metric Tons Per Capita", "Metric_Tons_Per_Capita")

# Since our data is in a Volume and volumes do not support delta transaction tables, convert the data to a managed table and then save as a delta table.
df_parquet.write.format("delta").mode("overwrite").saveAsTable(
    "databrickslinkedin.default.fortesting"
)

# Verify the Delta table
delta_table = spark.read.table("databrickslinkedin.default.fortesting")
display(delta_table)



In [0]:
# Total emissions Co2 by Country and Date
# Firsch check the date format:
display(df.select("Date").distinct().orderBy("Date", ascending=True))